# 05 · Conclusión de drift y decisión champion/candidate

## Objetivo

Consolidar la evidencia del cambio entre `group-5-batch-4` y `group-5-batch-5`, evitando conclusiones basadas en muestras no representativas. Este notebook es la referencia para la sustentación.

## Hallazgos de datos

- Los dos batches comparten los mismos 53 estados; no hay separación total por estado.
- Comparten 7.254 ciudades y 11.966 códigos postales, aunque cambia su composición.
- La muestra aleatoria detectó drift de entrada en `house_size`, `bath` y `bed`; `acre_lot` y `prev_sold_year` se mantuvieron estables.
- `zip_code`, `city` y `state` cambian en frecuencia, mientras `status` es estable.
- Batch 4 cubre aproximadamente precios entre 300k y 500k; batch 5 comienza por encima de 500k y tiene una cola de precios mucho mayor.

## Desempeño

| Conjunto | MAE | RMSE | R² |
|---|---:|---:|---:|
| Test de entrenamiento | 41.027,94 | — | — |
| `group-5-batch-4` | 40.677,62 | 48.774,13 | 0,2832 |
| `group-5-batch-5` | 941.216,05 | 2.159.470 | -0,2308 |

El champion funciona en el rango histórico, pero se degrada en el segmento alto. Esto prueba degradación de desempeño; no demuestra por sí solo concept drift estricto de la relación P(y|x).

## Candidates evaluados

Los candidates mejoraron batch 5, pero empeoraron batch 4. El candidate Ridge fue rechazado por predicciones extremas al invertir la transformación logarítmica. Ninguna alternativa cumplió una promoción global.

**Decisión:** conservar `property_price_model@champion` versión 1; no promover automáticamente. Para una futura promoción se requiere muestreo estratificado, validación de calidad/outliers y un candidate que cumpla métricas en ambos segmentos.

## Implicación para serving

La API carga el alias `champion` una vez durante su inicio y mantiene el modelo en memoria. No consulta MLflow en cada predicción. Una promoción futura requerirá actualizar el alias y reiniciar o hacer rollout de `property-api`; Streamlit y Locust consumirán automáticamente el modelo cargado por esa API.